# TextGraphicalizer: Aesop fables

Run the Laya-backed `TextGraphicalizer` on stories loaded from the cached
Project Gutenberg Aesop corpus using the checked-in high-level WordNet
ontology. The estimator owns the parameterized graph display function, so
rendering can be reused outside this notebook.

> The first model load downloads the pinned Laya checkpoint into the Hugging Face cache.
> The first corpus load downloads and caches the cleaned Aesop stories.


## One-time setup

Use the Python environment selected for this project. From the repository root, run these commands once in a terminal:

```bash
python -m pip install -e ".[notebook]"
python -m ipykernel install --user --name textgraphicalizer --display-name "TextGraphicalizer"
```

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import textwrap

assert (3, 10) <= sys.version_info[:2] <= (3, 12), (
    f"Use a Python 3.10–3.12 kernel; current interpreter is {sys.version}"
)
print(f"Using Python: {sys.executable}")

import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "ontology.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from textgraphicalizer import TextGraphicalizer, load_aesop_fables

plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 15, "font.size": 10})
ontology_path = ROOT / "fairy_tale_ontology.yaml"
stopwords_path = ROOT / "stopwords.yaml"
ontology_path

Using Python: /Users/f.costa/.venvs/py312/bin/python


PosixPath('/Users/f.costa/Code/TextGraphicalizer/fairy_tale_ontology.yaml')

In [ ]:
extractor = TextGraphicalizer(
    use_llm=True,
    llm_provider="mlx-lm", # 'openai' or 'ollama' or 'mlx-lm'
)

## Aesop stories

In [ ]:
stories = load_aesop_fables()
selected_stories = stories[1:2]
documents = [
    (story.split("\n\n", 1)[0], story)
    for story in selected_stories
]
def wrap_document(document, width=150):
    return "\n".join(
        textwrap.fill(
            line,
            width=width,
            break_long_words=False,
            break_on_hyphens=False,
        )
        for line in document.splitlines()
    )

print(f"Loaded {len(stories)} stories; graphicalizing {len(documents)}.")
for index, (title, document) in enumerate(documents, 1):
    print(f"\n--- Story {index}: {title} ---")
    print(wrap_document(document))

Loaded 284 stories; graphicalizing 2.

--- Story 1: THE GOOSE THAT LAID THE GOLDEN EGGS ---
THE GOOSE THAT LAID THE GOLDEN EGGS

A Man and his Wife had the good fortune to possess a Goose which laid a Golden Egg every day. Lucky though they were, they soon began to think they
were not getting rich fast enough, and, imagining the bird must be made of gold inside, they decided to kill it in order to secure the whole store of
precious metal at once. But when they cut it open they found it was just like any other goose. Thus, they neither got rich all at once, as they had
hoped, nor enjoyed any longer the daily addition to their wealth.

Much wants more and loses all.

--- Story 2: THE CAT AND THE MICE ---
THE CAT AND THE MICE

There was once a house that was overrun with Mice. A Cat heard of this, and said to herself, "That's the place for me," and off she went and took up
her quarters in the house, and caught the Mice one by one and ate them. At last the Mice could stand it no longer, an

In [4]:
extractor.ontology = ontology_path
extractor.stopwords_path = stopwords_path

In [5]:
extractor.node_threshold = 0.25
extractor.edge_threshold = 0.3
extractor.connected = True
extractor.max_node_degree = 3
extractor.use_milp = True

In [ ]:
from time import time
graphs = []
for i, (title, document) in enumerate(documents):
    wrapped_document = wrap_document(document)
    print(f"Document {i + 1}/{len(documents)}:\n{wrapped_document}")
    start_time = time()
    graph = extractor.transform(document)
    graphs.append((title, document, graph))
    end_time = time()
    elapsed_time = end_time - start_time
    print(f"{title}: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges (took {elapsed_time:.2f} seconds)")
    print("-" * 80)

Document 1/2:
THE GOOSE THAT LAID THE GOLDEN EGGS

A Man and his Wife had the good fortune to possess a Goose which laid a Golden Egg every day. Lucky though they were, they soon began to think they
were not getting rich fast enough, and, imagining the bird must be made of gold inside, they decided to kill it in order to secure the whole store of
precious metal at once. But when they cut it open they found it was just like any other goose. Thus, they neither got rich all at once, as they had
hoped, nor enjoyed any longer the daily addition to their wealth.

Much wants more and loses all.


In [ ]:
for title, document, graph in graphs:
    extractor.display(
        graph,
        title=title,
        document=document,
        max_char=150,
    )


In [ ]:
for title, document, graph in graphs:
    display(extractor.display_d3(
        graph,
        title=title,
        document=document,
        max_char=100,
    ))


## Inspect one graph as ordinary NetworkX data

In [ ]:
title, document, graph = graphs[0]
print("Story:", title)
print("Nodes:")
display(list(graph.nodes(data=True)))
print("Edges:")
display(list(graph.edges(data=True)))
print("Graph metadata:")
display(graph.graph)
